# CAG Parsing Pipeline Inspection Notebook

This notebook provides detailed inspection of the complete 3-phase CAG parsing pipeline:
**ManifestIngestionService → TriageService → ScaffoldingService**

We process all 5 pilot CAG audit reports and examine the output at each stage.

## Overview
- **Phase 1**: Manifest ingestion and PDF downloads
- **Phase 2**: Document triage (native vs scanned classification)  
- **Phase 3**: Document scaffolding (ToC extraction + page mappings)

## Reports Processed
1. **report_001**: CAG Report on Cleanliness and Sanitation in Indian Railways
2. **report_002**: Direct Taxes Audit Report - Union Government Revenue
3. **report_003**: CAG Report on Fiscal Responsibility (SCANNED)
4. **report_004**: Solar Parks and Ultra Mega Solar Power Projects
5. **report_005**: Indian National Centre for Ocean Information Services

## 1. Environment Setup

In [1]:
# Add source directory to path for imports
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

# Import required modules
import asyncio
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown, HTML

# Import our services
from modules.manifest_ingestion_service import ManifestIngestionService
from modules.triage_service import TriageService
from modules.scaffolding_service import ScaffoldingService
from modules.data_contracts import DocumentTask

# Set up display options
pd.set_option('display.max_colwidth', None)
pd.set_option('display.expand_frame_repr', False)

print("✅ Environment initialized")
print(f"📂 Current directory: {os.getcwd()}")
print(f"📂 Data directory: {Path('../data/raw').resolve()}")

✅ Environment initialized
📂 Current directory: /Users/dev/Desktop/Desktop - Dev’s MacBook Pro/Repositories/CAG/services/parsing_pipeline/notebooks
📂 Data directory: /Users/dev/Desktop/Desktop - Dev’s MacBook Pro/Repositories/CAG/services/parsing_pipeline/data/raw


## 2. Load and Inspect Manifest

In [2]:
# Load the manifest file
manifest_path = Path('../../../CAG-Union Audit Reports.xlsx').resolve()

if manifest_path.exists():
    print(f"✅ Manifest found: {manifest_path}")
    
    # Read and display the manifest
    manifest_df = pd.read_excel(manifest_path, header=1)
    print(f"📊 Manifest shape: {manifest_df.shape}")
    print(f"📋 Columns: {list(manifest_df.columns)}")
    
    # Display first few rows
    display(Markdown("### Manifest Preview"))
    display(manifest_df.head())
    
    # Check for existing PDFs
    raw_data_dir = Path('../data/raw')
    pdf_files = list(raw_data_dir.glob('*.pdf'))
    print(f"\n📄 Found {len(pdf_files)} PDF files:")
    for pdf in pdf_files:
        size_mb = pdf.stat().st_size / (1024 * 1024)
        print(f"  • {pdf.name}: {size_mb:.1f} MB")
        
else:
    print(f"❌ Manifest not found: {manifest_path}")

✅ Manifest found: /Users/dev/Desktop/Desktop - Dev’s MacBook Pro/Repositories/CAG/CAG-Union Audit Reports.xlsx
📊 Manifest shape: (5, 9)
📋 Columns: ['  SL NO', 'Date', 'Original Title', 'Recommended Title', 'Government Type', 'Union Department', 'Report Type', 'Sector', 'Report PDF']


### Manifest Preview

,SL NO,Date,Original Title,Recommended Title,Government Type,Union Department,Report Type,Sector,Report PDF
0,1,2025-08-20,"Report of the Comptroller and Auditor General of India for the period ended March 2023, Report No. 15 of 2025- Union Government (Railways) (Performance Audit) on ""Cleanliness and Sanitation in long distance trains in Indian Railways""",CAG Report on Cleanliness and Sanitation in Indian Railways,Union,Railways,Performance,Transport & Infrastructure,"https://cag.gov.in/webroot/uploads/download_audit_report/2025/Final_Report-No.-15-of-2025-(Railways)-ENG,digitize--signed-068a8522d0cd480.14636815.pdf"
1,2,2025-08-18,Report of the Comptroller and Auditor General of India on Direct Taxes for the period ended March 2023 Union Government Department of Revenue Report No. 14 of 2025 (Compliance Audit – Civil),Direct Taxes Audit Report: Union Government Revenue for March 2023,Union,Civil,Compliance,Taxes and Duties,https://cag.gov.in/webroot/uploads/download_audit_report/2025/Report-No.-14-of-2025_CA-2022-23_English-PDF-A-068a46a0b283d09.44138066.pdf
2,3,2025-08-18,"Report of the Comptroller and Auditor General of India on Compliance of the Fiscal Responsibility and Budget Management Act, 2003 for the year 2023-24 Union Government Department of Economic Affairs (Ministry of Finance) Report No. 19 of 2025",CAG Report on Fiscal Responsibility and Budget Management Act Compliance for 2023-24,Union,Civil,Financial,Finance,https://cag.gov.in/webroot/uploads/download_audit_report/2025/Report-No.-19-of-2025_Eng_DigitallySign-068a3f34618f311.84605350.pdf
3,4,2025-08-12,Report of the Comptroller and Auditor General of India on Solar Parks and Ultra Mega Solar Power Projects for the year 2017-22 Union Government Ministry of New and Renewable Energy Report No. 13 of 2025 (Performance Audit),Solar Parks and Ultra Mega Solar Power Projects: A 2017-2022 Performance Audit by the Comptroller and Auditor General of India,Union,Scientific Department,Performance,Power and Energy,https://cag.gov.in/webroot/uploads/download_audit_report/2025/Report-No.-13-of-2025_Solar-English-(16-07-2025)-0689dd00e3c73c7.54926606.pdf
4,5,2025-08-12,Report of the Comptroller and Auditor General of India on the Activities of Indian National Centre for Ocean Information Services Union Government Ministry of Earth Sciences Report No. 8 of 2025 (Compliance Audit-Civil),CAG Report on Indian National Centre for Ocean Information Services,Union,Scientific Department,Compliance,Science and Technology,https://cag.gov.in/webroot/uploads/download_audit_report/2025/Final-SSCA-to-hq-incois-english_signed-10-06-2025-0689dd1177c9832.61823377.pdf



📄 Found 0 PDF files:


## 3. Phase 1: ManifestIngestionService Inspection

In [3]:
# Initialize and run ManifestIngestionService
manifest_service = ManifestIngestionService(raw_data_dir='../data/raw')

# Process the manifest
async def process_manifest():
    tasks = await manifest_service.process_manifest(str(manifest_path))
    return tasks

# Run async function
manifest_tasks = await process_manifest()

print(f"📋 Phase 1 Results:")
print(f"   Total tasks created: {len(manifest_tasks)}")

# Separate successful and failed tasks
successful_tasks = [t for t in manifest_tasks if t.processing_status != 'failed_download']
failed_tasks = [t for t in manifest_tasks if t.processing_status == 'failed_download']

print(f"   ✅ Successful downloads: {len(successful_tasks)}")
print(f"   ❌ Failed downloads: {len(failed_tasks)}")

# Display failed tasks if any
if failed_tasks:
    print("\n🚫 Failed downloads:")
    for task in failed_tasks:
        print(f"   • {task.report_id}: {'; '.join(task.error_log)}")

# Display successful tasks with details
if successful_tasks:
    print("\n✅ Successful tasks:")
    success_data = []
    for task in successful_tasks:
        success_data.append({
            'Report ID': task.report_id,
            'Source URL': task.source_url[:50] + '...' if len(task.source_url) > 50 else task.source_url,
            'Local PDF': task.local_pdf_path,
            'File Size (MB)': round(Path(task.local_pdf_path).stat().st_size / (1024*1024), 2) if Path(task.local_pdf_path).exists() else 'N/A'
        })
    
    success_df = pd.DataFrame(success_data)
    display(success_df)
    
    # Show metadata for first task as example
    if successful_tasks:
        first_task = successful_tasks[0]
        display(Markdown(f"### Sample DocumentTask Metadata ({first_task.report_id})"))
        
        # Convert metadata to DataFrame for better display
        metadata_df = pd.DataFrame([
            {'Field': k, 'Value': str(v), 'Type': type(v).__name__}
            for k, v in first_task.initial_metadata.items()
        ])
        display(metadata_df)

Loaded with header=1, raw columns: ['  SL NO', 'Date', 'Original Title', 'Recommended Title', 'Government Type', 'Union Department', 'Report Type', 'Sector', 'Report PDF']
Standardized columns: ['SL NO', 'Date', 'Title', 'Recommended Title', 'Government Type', 'Union Department', 'Report Type', 'Sector', 'Report PDF']
Ingestion complete: 5 downloads successful, 0 failed.
📋 Phase 1 Results:
   Total tasks created: 5
   ✅ Successful downloads: 5
   ❌ Failed downloads: 0

✅ Successful tasks:


,Report ID,Source URL,Local PDF,File Size (MB)
0,report_001_cag_report_on_cleanliness_and_sanitation_in_indian,https://cag.gov.in/webroot/uploads/download_audit_...,../data/raw/report_001_cag_report_on_cleanliness_and_sanitation_in_indian.pdf,5.43
1,report_002_direct_taxes_audit_report_union_government_revenu,https://cag.gov.in/webroot/uploads/download_audit_...,../data/raw/report_002_direct_taxes_audit_report_union_government_revenu.pdf,2.09
2,report_003_cag_report_on_fiscal_responsibility_and_budget_man,https://cag.gov.in/webroot/uploads/download_audit_...,../data/raw/report_003_cag_report_on_fiscal_responsibility_and_budget_man.pdf,1.48
3,report_004_solar_parks_and_ultra_mega_solar_power_projects_a,https://cag.gov.in/webroot/uploads/download_audit_...,../data/raw/report_004_solar_parks_and_ultra_mega_solar_power_projects_a.pdf,1.87
4,report_005_cag_report_on_indian_national_centre_for_ocean_inf,https://cag.gov.in/webroot/uploads/download_audit_...,../data/raw/report_005_cag_report_on_indian_national_centre_for_ocean_inf.pdf,4.33


### Sample DocumentTask Metadata (report_001_cag_report_on_cleanliness_and_sanitation_in_indian)

,Field,Value,Type
0,SL NO,1,int
1,Date,2025-08-20 00:00:00,Timestamp
2,Title,"Report of the Comptroller and Auditor General of India for the period ended March 2023, Report No. 15 of 2025- Union Government (Railways) (Performance Audit) on ""Cleanliness and Sanitation in long distance trains in Indian Railways""",str
3,Recommended Title,CAG Report on Cleanliness and Sanitation in Indian Railways,str
4,Government Type,Union,str
5,Union Department,Railways,str
6,Report Type,Performance,str
7,Sector,Transport & Infrastructure,str
8,Report PDF,"https://cag.gov.in/webroot/uploads/download_audit_report/2025/Final_Report-No.-15-of-2025-(Railways)-ENG,digitize--signed-068a8522d0cd480.14636815.pdf",str


## 4. Phase 2: TriageService Inspection

In [4]:
# Initialize TriageService
triage_service = TriageService()

# Process the successful tasks from Phase 1
triaged_native = []
triaged_scanned = []
failed_triage = []
triage_results = []

print("🔍 Phase 2: Document Triage Analysis")
print("=" * 50)

for i, task in enumerate(successful_tasks, 1):
    print(f"\n📄 [{i:2d}/ {len(successful_tasks)}] Triaging: {task.report_id}")
    
    # Run triage
    triaged_task = triage_service.triage_document(task)
    
    # Store results
    if triaged_task.classification == 'native_text':
        triaged_native.append(triaged_task)
        classification = "NATIVE TEXT"
        symbol = "📝"
    elif triaged_task.classification == 'scanned':
        triaged_scanned.append(triaged_task)
        classification = "SCANNED IMAGE"
        symbol = "📷"
    else:
        failed_triage.append(triaged_task)
        classification = "FAILED"
        symbol = "❌"
    
    # Display results
    print(f"         {symbol} Classification: {classification}")
    if hasattr(triaged_task, 'error_log') and triaged_task.error_log:
        print(f"         ⚠️  Error: {triaged_task.error_log[-1]}")
    
    # Store for summary
    triage_results.append({
        'Report ID': task.report_id,
        'Classification': classification,
        'Status': triaged_task.processing_status,
        'Error': '; '.join(triaged_task.error_log) if triaged_task.error_log else ''
    })

# Summary statistics
print("\n" + "=" * 50)
print("PHASE 2 SUMMARY")
print("=" * 50)
print(f"Total documents processed: {len(successful_tasks)}")
print(f"📝 Native text documents: {len(triaged_native)}")
print(f"📷 Scanned documents: {len(triaged_scanned)}")
print(f"❌ Failed triage: {len(failed_triage)}")

# Display results table
triage_df = pd.DataFrame(triage_results)
display(Markdown("### Triage Results Table"))
display(triage_df)

# Show samples
if triaged_native:
    display(Markdown(f"### 📝 Native Text Sample ({triaged_native[0].report_id})"))
    display(Markdown(f"Ready for direct LayoutAnalysis processing"))
    
if triaged_scanned:
    display(Markdown(f"### 📷 Scanned Document Sample ({triaged_scanned[0].report_id})"))
    display(Markdown(f"Will require OCRService preprocessing"))

🔍 Phase 2: Document Triage Analysis

📄 [ 1/ 5] Triaging: report_001_cag_report_on_cleanliness_and_sanitation_in_indian
         📝 Classification: NATIVE TEXT

📄 [ 2/ 5] Triaging: report_002_direct_taxes_audit_report_union_government_revenu
         📝 Classification: NATIVE TEXT

📄 [ 3/ 5] Triaging: report_003_cag_report_on_fiscal_responsibility_and_budget_man
         📷 Classification: SCANNED IMAGE

📄 [ 4/ 5] Triaging: report_004_solar_parks_and_ultra_mega_solar_power_projects_a
         📝 Classification: NATIVE TEXT

📄 [ 5/ 5] Triaging: report_005_cag_report_on_indian_national_centre_for_ocean_inf
         📝 Classification: NATIVE TEXT

PHASE 2 SUMMARY
Total documents processed: 5
📝 Native text documents: 4
📷 Scanned documents: 1
❌ Failed triage: 0


### Triage Results Table

,Report ID,Classification,Status,Error
0,report_001_cag_report_on_cleanliness_and_sanitation_in_indian,NATIVE TEXT,triaged_native,
1,report_002_direct_taxes_audit_report_union_government_revenu,NATIVE TEXT,triaged_native,
2,report_003_cag_report_on_fiscal_responsibility_and_budget_man,SCANNED IMAGE,triaged_scanned,
3,report_004_solar_parks_and_ultra_mega_solar_power_projects_a,NATIVE TEXT,triaged_native,
4,report_005_cag_report_on_indian_national_centre_for_ocean_inf,NATIVE TEXT,triaged_native,


### 📝 Native Text Sample (report_001_cag_report_on_cleanliness_and_sanitation_in_indian)

Ready for direct LayoutAnalysis processing

### 📷 Scanned Document Sample (report_003_cag_report_on_fiscal_responsibility_and_budget_man)

Will require OCRService preprocessing

## 5. Phase 3: ScaffoldingService Inspection

In [5]:
# Initialize ScaffoldingService
scaffolding_service = ScaffoldingService()

# Prepare all successfully triaged tasks
all_triaged = triaged_native + triaged_scanned
scaffold_results = []
scaffold_complete = []
scaffold_minimal = []
scaffold_partial = []
scaffold_failed = []

print("🏗️  Phase 3: Document Scaffolding Analysis")
print("=" * 50)

for i, task in enumerate(all_triaged, 1):
    print(f"\n🏗️  [{i:2d}/ {len(all_triaged)}] Scaffolding: {task.report_id}")
    
    # Build scaffold
    scaffolded_task = scaffolding_service.build_scaffold(task)
    
    # Analyze scaffold
    toc_entries = len(scaffolded_task.scaffold.get('toc', []))
    page_mappings = len(scaffolded_task.scaffold.get('page_map', {}))
    
    # Display results
    print(f"         Status: {scaffolded_task.processing_status}")
    print(f"         ToC Entries: {toc_entries}")
    print(f"         Page Mappings: {page_mappings}")
    
    if scaffolded_task.error_log:
        print(f"         Log: {'; '.join(scaffolded_task.error_log)}")
    
    # Categorize results
    if scaffolded_task.processing_status == 'scaffold_complete':
        scaffold_complete.append(scaffolded_task)
        category = "COMPLETE (ToC + Pages)"
    elif scaffolded_task.processing_status == 'scaffold_minimal':
        scaffold_minimal.append(scaffolded_task)
        category = "MINIMAL (ToC only)"
    elif scaffolded_task.processing_status == 'scaffold_partial':
        scaffold_partial.append(scaffolded_task)
        category = "PARTIAL (Pages only)"
    else:
        scaffold_failed.append(scaffolded_task)
        category = "FAILED"
    
    # Store results
    scaffold_results.append({
        'Report ID': task.report_id,
        'Classification': 'Native' if task in triaged_native else 'Scanned',
        'Status': category,
        'ToC Entries': toc_entries,
        'Page Mappings': page_mappings,
        'Errors': '; '.join(scaffolded_task.error_log) if scaffolded_task.error_log else ''
    })

# Summary statistics
print("\n" + "=" * 50)
print("PHASE 3 SUMMARY")
print("=" * 50)
print(f"Total documents scaffolded: {len(all_triaged)}")
print(f"🟢 Complete scaffolds: {len(scaffold_complete)}")
print(f"🟡 Minimal scaffolds: {len(scaffold_minimal)}")
print(f"🟠 Partial scaffolds: {len(scaffold_partial)}")
print(f"🔴 Failed scaffolds: {len(scaffold_failed)}")

# Display results table
scaffold_df = pd.DataFrame(scaffold_results)
display(Markdown("### Scaffolding Results Table"))
display(scaffold_df)

# Show detailed scaffold for complete examples
if scaffold_complete:
    sample_complete = scaffold_complete[0]
    display(Markdown(f"### 🟢 Complete Scaffold Example: {sample_complete.report_id}"))
    
    # Show ToC entries
    toc = sample_complete.scaffold.get('toc', [])
    if toc:
        display(Markdown("**ToC Preview (first 10 entries):**"))
        toc_preview = pd.DataFrame([
            {'Level': entry[0], 'Title': entry[1][:100] + '...' if len(entry[1]) > 100 else entry[1], 'Page': entry[2]}
            for entry in toc[:10]
        ])
        display(toc_preview)
    
    # Show page mappings
    page_map = sample_complete.scaffold.get('page_map', {})
    if page_map:
        display(Markdown("**Page Mappings Preview (first 15 mappings):**"))
        page_preview = pd.DataFrame([
            {'Physical Page': k, 'Logical Label': v}
            for k, v in list(page_map.items())[:15]
        ])
        display(page_preview)

🏗️  Phase 3: Document Scaffolding Analysis

🏗️  [ 1/ 5] Scaffolding: report_001_cag_report_on_cleanliness_and_sanitation_in_indian
         Status: scaffold_complete
         ToC Entries: 119
         Page Mappings: 128
         Log: Embedded ToC extracted successfully: 119 entries; Page mappings created: 128 entries

🏗️  [ 2/ 5] Scaffolding: report_002_direct_taxes_audit_report_union_government_revenu
         Status: scaffold_complete
         ToC Entries: 166
         Page Mappings: 136
         Log: Embedded ToC inadequate: 6 entries, proceeding to heuristic generation; Heuristic ToC generated with 166 entries; Page mappings created: 136 entries

🏗️  [ 3/ 5] Scaffolding: report_004_solar_parks_and_ultra_mega_solar_power_projects_a
         Status: scaffold_complete
         ToC Entries: 235
         Page Mappings: 92
         Log: Embedded ToC inadequate: 2 entries, proceeding to heuristic generation; Heuristic ToC generated with 235 entries; Page mappings created: 92 entries

🏗️  

### Scaffolding Results Table

,Report ID,Classification,Status,ToC Entries,Page Mappings,Errors
0,report_001_cag_report_on_cleanliness_and_sanitation_in_indian,Native,COMPLETE (ToC + Pages),119,128,Embedded ToC extracted successfully: 119 entries; Page mappings created: 128 entries
1,report_002_direct_taxes_audit_report_union_government_revenu,Native,COMPLETE (ToC + Pages),166,136,"Embedded ToC inadequate: 6 entries, proceeding to heuristic generation; Heuristic ToC generated with 166 entries; Page mappings created: 136 entries"
2,report_004_solar_parks_and_ultra_mega_solar_power_projects_a,Native,COMPLETE (ToC + Pages),235,92,"Embedded ToC inadequate: 2 entries, proceeding to heuristic generation; Heuristic ToC generated with 235 entries; Page mappings created: 92 entries"
3,report_005_cag_report_on_indian_national_centre_for_ocean_inf,Native,COMPLETE (ToC + Pages),204,158,"Embedded ToC inadequate: 12 entries, proceeding to heuristic generation; Heuristic ToC generated with 204 entries; Page mappings created: 158 entries"
4,report_003_cag_report_on_fiscal_responsibility_and_budget_man,Scanned,COMPLETE (ToC + Pages),62,68,Embedded ToC extracted successfully: 62 entries; Page mappings created: 68 entries


### 🟢 Complete Scaffold Example: report_001_cag_report_on_cleanliness_and_sanitation_in_indian

**ToC Preview (first 10 entries):**

,Level,Title,Page
0,1,Report No. 15 of 2025 (Railways) ENG cover - Only,1
1,1,"Final_Report No. 15 of 2025 (Railways) ENG,digitize",3
2,2,Table of Contents,7
3,2,Executive Summary,11
4,2,Chapter I: Introduction,19
5,3,1. Background,21
6,4,1.1 Organisational Structure,22
7,4,1.2 Audit approach,23
8,5,1.2.1 Audit objectives,23
9,5,1.2.2 Scope of audit,24


**Page Mappings Preview (first 15 mappings):**

,Physical Page,Logical Label
0,0,1
1,1,2
2,2,3
3,3,4
4,4,5
5,5,6
6,6,7
7,7,8
8,8,9
9,9,10


## 6. Cross-Service Analysis

In [6]:
# Create comprehensive analysis combining all phases
print("📊 CROSS-SERVICE ANALYSIS")
print("=" * 50)

# Combine all results
final_tasks = scaffold_complete + scaffold_minimal + scaffold_partial + scaffold_failed

comprehensive_results = []
for task in final_tasks:
    # Find corresponding triage results
    classification = "Unknown"
    if task in triaged_native:
        classification = "Native Text"
    elif task in triaged_scanned:
        classification = "Scanned Image"
    
    # Scaffold status
    scaffold_status = task.processing_status.replace('scaffold_', '').title()
    
    # Metrics
    toc_entries = len(task.scaffold.get('toc', []))
    page_mappings = len(task.scaffold.get('page_map', {}))
    file_size_mb = round(Path(task.local_pdf_path).stat().st_size / (1024*1024), 2) if Path(task.local_pdf_path).exists() else 0
    
    comprehensive_results.append({
        'Report ID': task.report_id,
        'PDF Size (MB)': file_size_mb,
        'Downloaded': '✅',
        'Triage': classification,
        'Scaffold': scaffold_status,
        'ToC Entries': toc_entries,
        'Page Maps': page_mappings,
        'Ready for Next': '✅ LayoutAnalysis' if classification == 'Native Text' else '⚠️  Needs OCR'
    })

# Display comprehensive results
final_df = pd.DataFrame(comprehensive_results)
display(Markdown("### 📋 Comprehensive Pipeline Results"))
display(final_df)

# Generate summary statistics
total_processed = len(final_df)
successful_e2e = len(final_df[
    (final_df['Downloaded'] == '✅') & 
    ((final_df['Scaffold'] == 'Complete') | (final_df['Scaffold'] == 'Minimal') | (final_df['Scaffold'] == 'Partial'))
])
ready_for_next = len(final_df[final_df['Triage'] == 'Native Text'])
scanned_count = len(final_df[final_df['Triage'] == 'Scanned Image'])

print(f"\n🎯 PIPELINE PERFORMANCE SUMMARY")
print(f"Total reports processed: {total_processed}")
print(f"End-to-end success rate: {successful_e2e}/{total_processed} ({successful_e2e/total_processed*100:.1f}%)")
print(f"Ready for LayoutAnalysis: {ready_for_next}")
print(f"Will need OCR processing: {scanned_count}")
print(f"Structural metadata generated: {sum(final_df['ToC Entries'])} ToC entries, {sum(final_df['Page Maps'])} page mappings")

# Show next steps based on results
display(Markdown("### 🎯 Next Steps & Recommendations"))

if ready_for_next > 0:
    display(Markdown(f"**✅ {ready_for_next} documents** are ready for immediate LayoutAnalysisService integration"))
    
if scanned_count > 0:
    display(Markdown(f"**⚠️  {scanned_count} document** requires OCRService development before LayoutAnalysis"))
    
total_toc = sum(final_df['ToC Entries'])
avg_toc_per_doc = total_toc / total_processed if total_processed > 0 else 0
display(Markdown(f"**📚 Structural Metadata**: Average {avg_toc_per_doc:.1f} ToC entries per document with complete page reconciliation"))

📊 CROSS-SERVICE ANALYSIS


### 📋 Comprehensive Pipeline Results

,Report ID,PDF Size (MB),Downloaded,Triage,Scaffold,ToC Entries,Page Maps,Ready for Next
0,report_001_cag_report_on_cleanliness_and_sanitation_in_indian,5.43,✅,Native Text,Complete,119,128,✅ LayoutAnalysis
1,report_002_direct_taxes_audit_report_union_government_revenu,2.09,✅,Native Text,Complete,166,136,✅ LayoutAnalysis
2,report_004_solar_parks_and_ultra_mega_solar_power_projects_a,1.87,✅,Native Text,Complete,235,92,✅ LayoutAnalysis
3,report_005_cag_report_on_indian_national_centre_for_ocean_inf,4.33,✅,Native Text,Complete,204,158,✅ LayoutAnalysis
4,report_003_cag_report_on_fiscal_responsibility_and_budget_man,1.48,✅,Scanned Image,Complete,62,68,⚠️ Needs OCR



🎯 PIPELINE PERFORMANCE SUMMARY
Total reports processed: 5
End-to-end success rate: 5/5 (100.0%)
Ready for LayoutAnalysis: 4
Will need OCR processing: 1
Structural metadata generated: 786 ToC entries, 582 page mappings


### 🎯 Next Steps & Recommendations

**✅ 4 documents** are ready for immediate LayoutAnalysisService integration

**⚠️  1 document** requires OCRService development before LayoutAnalysis

**📚 Structural Metadata**: Average 157.2 ToC entries per document with complete page reconciliation

## 7. Export Results for Further Analysis

In [ ]:
# Export comprehensive results to CSV for external analysis
output_dir = Path('../notebooks/results')
output_dir.mkdir(exist_ok=True)

# Export detailed results
final_df.to_csv(output_dir / 'cag_pipeline_full_results.csv', index=False)

# Export scaffold details for each document
for task in final_tasks:
    task_results = {
        'report_id': task.report_id,
        'pdf_path': task.local_pdf_path,
        'classification': task.classification,
        'processing_status': task.processing_status,
        'toc_entries': len(task.scaffold.get('toc', [])),
        'page_mappings': len(task.scaffold.get('page_map', [])),
        'toc_data': task.scaffold.get('toc', []),
        'page_map_data': task.scaffold.get('page_map', {}),
        'error_log': task.error_log
    }
    
    # Save individual task results
    task_filename = f"{task.report_id}_scaffold_results.json"
    import json
    with open(output_dir / task_filename, 'w') as f:
        # Convert unserializable objects to strings
        serializable_results = task_results.copy()
        serializable_results['toc_data'] = [
            {'level': entry[0], 'title': entry[1], 'page': entry[2]}
            for entry in serializable_results['toc_data']
        ]
        json.dump(serializable_results, f, indent=2, default=str)

print("💾 Results exported to:")
print(f"   📊 {output_dir / 'cag_pipeline_full_results.csv'}")
print(f"   📄 Individual JSON files in {output_dir}/")
print("\n📈 Ready for external analysis and visualization!")